<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>Sharif University of Technology</b></p>
    <p>Deep Learning Course, Dr. Soleymani</p>
    <p>Spring 2026</p>
</div>

---


*Full Name:* Shayan Salehi

*Student ID:* 404203509

# Retrieval-Augmented Generation (RAG) – From Scratch to Conversation

## Overview

**Retrieval-Augmented Generation (RAG)** is a technique that enhances a language model’s output by first retrieving relevant information from a knowledge base, then conditioning the generation on that retrieved evidence.

Mathematically, given a query $x$, we want to model:

$
p(y \mid x) = \sum_{z \in \text{Top-}k(x)} p_{\text{retrieve}}(z \mid x) \; p_{\text{generate}}(y \mid x, z)
$

where $z$ is a retrieved document chunk, often treated as a hard selection after top‑k retrieval.

### Dataset used in this notebook

Instead of a manually written toy knowledge base, this version uses a small real subset of **SQuAD v1** from Hugging Face. SQuAD provides real Wikipedia-based contexts, questions, and gold answers. To keep the notebook lightweight, we only use a small validation subset.

### In this notebook you will:

1. **From scratch (no LangChain)**:
   - Load a small real QA dataset
   - Chunk real Wikipedia contexts
   - Create dense embeddings with SentenceTransformers
   - Build a FAISS index for fast nearest-neighbour search
   - Implement a retriever and a GPT‑2 generator
   - Assemble a complete, stateless RAG system

2. **With LangChain**:
   - Integrate memory for multi-turn conversations
   - Implement a **history-aware** retriever that rewrites follow-up questions

3. **With an encoder-decoder generator**:
   - Replace GPT‑2 with FLAN‑T5
   - Compare decoder-only RAG and encoder-decoder RAG on the same retrieved contexts

After completing this notebook, you will understand the core components of RAG, the need for memory in conversational systems, and the architectural difference between decoder-only and encoder-decoder generators.

**Dependencies**: Run the cell below to install required packages.

In [1]:
# Install necessary packages (if not already installed)
import sys
!{sys.executable} -m pip install --quiet torch transformers sentence-transformers faiss-cpu datasets langchain-community langchain-huggingface langchain-core
# We use:
# - datasets for loading a small real SQuAD subset
# - sentence-transformers for dense embeddings
# - faiss-cpu for vector search
# - transformers for GPT-2 and FLAN-T5 generators
# - langchain-community / langchain-huggingface for the vectorstore wrapper in the conversational part


In [4]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3'

import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA A100-SXM4-80GB


In [5]:
import os
import json
import torch
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Part 1: Document Loading & Text Chunking

A RAG system needs a collection of documents to retrieve from. In real applications these are often large texts, so we must split them into manageable **chunks**. Each chunk is a unit that can be embedded and later retrieved.

In this notebook, we use a small real subset of **SQuAD v1** instead of a manually written toy text file. Each SQuAD example contains:

- a real Wikipedia paragraph as `context`,
- a `question`,
- one or more gold `answers`.

We use the contexts as the retrieval corpus and the questions/gold answers for testing.

**Why chunk?**
- Language models have a limited context window.
- Smaller chunks allow more precise retrieval; a long document may contain many topics.

**Chunking trade‑offs**:
- Too small → loss of surrounding context.
- Too large → retrieval may bring irrelevant detail, and generation may exceed token limits.

**Implementation**: We’ll implement a simple recursive character‑based splitter with user‑defined chunk size and overlap.

### Questions 1
1. What would happen if we used whole documents without chunking?  
2. How does the overlap parameter help preserve continuity between chunks?

### Answers 1

1. If we used whole documents, a long context could be too big for the model's context window, and retrieval would be less precise because one document often covers many topics, so we would pull in a lot of text that is not relevant to the question.

2. Overlap repeats a small piece of text at the boundary between two chunks. This way a sentence or idea that falls right on the edge of one chunk is not cut in half, since it still appears whole inside the neighboring chunk.

In [6]:
# TODO: Implement `chunk_text`.
# Split text into overlapping character chunks and return a list of strings.
# Expected output is kept below as a reference.
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    if not text:
        return []
    text = text.strip()
    if not text:
        return []
    if chunk_size <= 0:
        return [text]
    if overlap < 0:
        overlap = 0
    if overlap >= chunk_size:
        overlap = chunk_size // 2

    chunks = []
    start = 0
    n = len(text)
    step = chunk_size - overlap
    while start < n:
        end = min(start + chunk_size, n)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= n:
            break
        start += step
    return chunks


sample = "Retrieval augmented generation combines a retriever and a generator. " * 5
demo = chunk_text(sample, chunk_size=120, overlap=20)
print(f"Number of chunks: {len(demo)}")
for c in demo[:3]:
    print(repr(c[:60]), "...")

Number of chunks: 4
'Retrieval augmented generation combines a retriever and a ge' ...
'combines a retriever and a generator. Retrieval augmented ge' ...
'rator. Retrieval augmented generation combines a retriever a' ...


In [8]:
# TODO: Load a small real SQuAD subset using `load_dataset`.
# Then extract unique contexts and create `chunks` using `chunk_text`.
# Expected output is kept below as a reference.
from datasets import load_dataset

squad = load_dataset("rajpurkar/squad", split="validation[:30]")

contexts = []
seen_ctx = set()
for ex in squad:
    ctx = ex["context"].strip()
    if ctx and ctx not in seen_ctx:
        seen_ctx.add(ctx)
        contexts.append(ctx)

qa_dataset = []
seen_q = set()
for ex in squad:
    q = ex["question"].strip()
    if q and q not in seen_q:
        seen_q.add(q)
        gold = ex["answers"]["text"][0] if ex["answers"]["text"] else ""
        qa_dataset.append({"question": q, "answer": gold, "context": ex["context"]})

chunks = []
for ctx in contexts:
    chunks.extend(chunk_text(ctx, chunk_size=300, overlap=50))

print(f"Unique contexts: {len(contexts)}")
print(f"Total chunks: {len(chunks)}")
print(f"QA pairs: {len(qa_dataset)}")
print("\nExample question:", qa_dataset[0]["question"])
print("Gold answer:", qa_dataset[0]["answer"])
print("\nExample chunk:\n", chunks[0][:200])

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Unique contexts: 1
Total chunks: 3
QA pairs: 27

Example question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Example chunk:
 Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated


## Part 2: Document Embeddings & Vector Index

To retrieve relevant chunks, we need to embed both the documents and the query into a common dense vector space. We use a pre‑trained sentence transformer to obtain **fixed‑size embeddings**.

**Dense retrieval** computes:

$
\text{sim}(q, d) = \cos(\phi(q), \phi(d)) \quad \text{or} \quad \|\phi(q) - \phi(d)\|_2
$

where $\phi$ is the embedding function.

We’ll store the embeddings in a **FAISS** index for fast approximate (or exact) nearest‑neighbour search.

### Questions 2
1. Explain the difference between sparse retrieval (e.g., TF‑IDF, BM25) and dense retrieval.  
2. Why is L2 distance often used instead of cosine similarity in FAISS `IndexFlatL2`? (Hint: embedding normalisation)

### Answers 2

1. Sparse retrieval such as TF-IDF or BM25 matches exact words and scores documents by how many query terms they share, so it struggles when the query and the document use different words for the same idea. Dense retrieval maps text into learned vectors, so it can match by meaning even when the wording is different.

2. FAISS IndexFlatL2 only knows how to compute L2 distance. If you first normalize the embeddings to unit length, the L2 distance becomes a simple decreasing function of the cosine similarity, so ranking by smallest L2 distance gives the same order as ranking by largest cosine. That is why people normalize the vectors and then just use L2.

In [9]:
# TODO: Load `all-MiniLM-L6-v2` with SentenceTransformer.
# Encode all chunks into float32 dense embeddings.
# Expected output is kept below as a reference.
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(device))

chunk_embeddings = embed_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=32,
).astype("float32")

print("Embeddings shape:", chunk_embeddings.shape)
print("Embeddings dtype:", chunk_embeddings.dtype)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (3, 384)
Embeddings dtype: float32


In [10]:
# TODO: Build a FAISS IndexFlatL2 index.
# Add all chunk embeddings to the index.
# Expected output is kept below as a reference.
import faiss

embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(chunk_embeddings)

print("Index dimension:", embedding_dim)
print("Vectors in index:", index.ntotal)

Index dimension: 384
Vectors in index: 3


## Part 3: The Retriever

The retriever is responsible for, given a user query, finding the top‑k most relevant chunks.

Formally:

$
\text{Retrieve}(q, k) = \text{argtopk}_{d \in \mathcal{D}} \; \text{sim}(\phi(q), \phi(d))
$

where $\mathcal{D}$ is the set of all chunk embeddings.

### Question 3
How would you modify the retriever to use **maximum inner product search** (MIPS) instead of L2 distance? When would MIPS be preferred?

### Answer 3

To use maximum inner product search you would build an inner-product index such as faiss.IndexFlatIP and search with the raw, un-normalized embeddings, so the score is the dot product instead of a distance. MIPS is preferred when the magnitude of the vectors carries useful information, for example when the embeddings are not normalized or when the model was trained with a dot-product objective.

In [11]:
# TODO: Implement `retrieve(query, index, embed_model, chunks, top_k)`.
# Embed the query, search FAISS, and return top-k (chunk, distance) pairs.
# Expected output is kept below as a reference.
def retrieve(query: str, index, embed_model, chunks, top_k: int = 3) -> List[Tuple[str, float]]:
    if not query or top_k <= 0 or len(chunks) == 0:
        return []
    top_k = min(top_k, len(chunks))
    query_embedding = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        if 0 <= idx < len(chunks):
            results.append((chunks[int(idx)], float(dist)))
    return results


sample_query = qa_dataset[0]["question"]
for chunk, dist in retrieve(sample_query, index, embed_model, chunks, top_k=3):
    print(f"[dist={dist:.3f}] {chunk[:90]}...")

[dist=0.630] Super Bowl 50 was an American football game to determine the champion of the National Foot...
[dist=1.035] arolina Panthers 24–10 to earn their third Super Bowl title. The game was played on Februa...
[dist=1.154] ersary" with various gold-themed initiatives, as well as temporarily suspending the tradit...


## Part 4: The Generator (GPT‑2)

We use a pre‑trained auto‑regressive language model (GPT‑2) to generate an answer conditioned on the retrieved context and the question.

**Input format** we will use:

Context:

chunk 1

chunk 2
...

Question: {user query}

Answer:

GPT‑2 will then continue the string.

### Questions 4
1. Why do we concatenate the context and the question?  
2. What are the limitations of using a base GPT‑2 model for knowledge‑intensive generation?

In [12]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt2_model_name = "gpt2"  # You can also try "gpt2-medium" if resources allow.
tokenizer = GPT2Tokenizer.from_pretrained(gpt2_model_name)
model = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
model.to(device)
model.eval()

# GPT-2 has no pad token by default; set it to eos for generation.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [13]:
# TODO: Implement `build_rag_prompt` and `generate_answer` for GPT-2.
# Concatenate retrieved context and question, then generate an answer.
# Expected output is kept below as a reference.
def build_rag_prompt(context_chunks: List[str], question: str) -> str:
    context = "\n\n".join(context_chunks) if context_chunks else "(no context retrieved)"
    return f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"


def generate_answer(prompt: str, max_new_tokens: int = 80) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    answer = text.split("\n")[0].strip()
    return answer if answer else text.strip()


_chunks = [c for c, _ in retrieve(qa_dataset[0]["question"], index, embed_model, chunks, top_k=3)]
_prompt = build_rag_prompt(_chunks, qa_dataset[0]["question"])
print("Q:", qa_dataset[0]["question"])
print("A:", generate_answer(_prompt, max_new_tokens=60))

Q: Which NFL team represented the AFC at Super Bowl 50?
A: The Denver Broncos.


## Part 5: Putting It All Together – Stateless RAG

Now we combine the retriever and the generator into a single pipeline.

Given a question:
1. Retrieve top‑k chunks.
2. Concatenate them into a context string.
3. Feed context + question to the GPT‑2 generator.

Let’s implement a `RAGSystem` class that encapsulates this process.

### Question 5
This system answers each question independently. What problem arises when a user asks a follow‑up question like *“Tell me more about that”* or *“Multiply the previous answer by 2”*?

### Answer 5

The system answers every question on its own and keeps no memory, so a follow-up like "tell me more about that" has no clear subject. The retriever only sees the new question, cannot tell what "that" or "the previous answer" refers to, and ends up retrieving the wrong chunks or failing to retrieve anything useful.

In [14]:
# TODO: Implement the stateless `RAGSystem` class.
# It should retrieve chunks, build the context, and generate an answer.
# Expected output is kept below as a reference.
class RAGSystem:
    def __init__(self, index, embed_model, chunks, tokenizer, model):
        self.index = index
        self.embed_model = embed_model
        self.chunks = chunks
        self.tokenizer = tokenizer
        self.model = model

    def query(self, question: str, top_k: int = 3, max_new_tokens: int = 80) -> Dict:
        retrieved = retrieve(question, self.index, self.embed_model, self.chunks, top_k=top_k)
        context_chunks = [c for c, _ in retrieved]
        prompt = build_rag_prompt(context_chunks, question)
        answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
        return {
            "question": question,
            "answer": answer,
            "retrieved_chunks": context_chunks,
            "distances": [d for _, d in retrieved],
        }


rag = RAGSystem(index, embed_model, chunks, tokenizer, model)

_test = qa_dataset[0]["question"]
_res = rag.query(_test, top_k=3)
print("Q:", _test)
print("A:", _res["answer"])
print("Num retrieved chunks:", len(_res["retrieved_chunks"]))

Q: Which NFL team represented the AFC at Super Bowl 50?
A: The Denver Broncos.
Num retrieved chunks: 3


## Part 6: The Memory Problem

Let’s simulate a conversation where the second question depends on the first one.

**Stateless RAG will fail or become unstable** because each call is independent. The retriever only sees the latest question, so a follow-up such as *“Where was it played?”* or *“Which team represented the NFC in that game?”* is ambiguous unless the system remembers that the previous topic was **Super Bowl 50**.


In [15]:
q1 = qa_dataset[0]["question"]
ans1 = rag.query(q1, top_k=3, max_new_tokens=80)
print(f"Q: {q1}\nA: {ans1['answer']}\n")

# Follow-up that refers to the previous topic.
# Without memory, the phrase "that game" is ambiguous for the retriever.
q2 = "Which team represented the NFC in that game?"
ans2 = rag.query(q2, top_k=3, max_new_tokens=80)
print(f"Q: {q2}\nA: {ans2['answer']}\n")

# Notice: the second answer may be weaker because stateless RAG does not explicitly
# remember that the previous question was about Super Bowl 50.

Q: Which NFL team represented the AFC at Super Bowl 50?
A: The Denver Broncos.

Q: Which team represented the NFC in that game?
A: The New York Giants.



## Part 7: Adding Memory with LangChain

To enable multi-turn conversations, we need to **maintain a chat history** and use it to reformulate the current question into a standalone query.

In many LangChain tutorials, this is done with functions such as `create_history_aware_retriever` and `create_retrieval_chain`. However, these imports can break across different LangChain versions. To keep this notebook stable, we implement the same logic explicitly while still using LangChain for the FAISS vectorstore wrapper.

We will:

1. Wrap the same FAISS index and embedding model with LangChain’s `FAISS` vectorstore.
2. Store the conversation in a simple `chat_history` list.
3. Rewrite ambiguous follow-up questions into standalone questions.
4. Retrieve documents using the standalone question.
5. Generate the final answer using **FLAN-T5**, which is much better for instruction following than base GPT-2.

### Questions 6
1. Explain how a history-aware retriever works internally. What prompt or rewriting step does it use?  
2. Why can’t we simply pass the whole chat history to the retriever instead of generating a standalone question?


In [16]:
# TODO: Create LangChain `Document` objects from chunks.
# Then build a FAISS vectorstore using HuggingFace embeddings.
# Expected output is kept below as a reference.
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS as LangChainFAISS
from langchain_huggingface import HuggingFaceEmbeddings

documents = [
    Document(page_content=chunk, metadata={"chunk_id": i})
    for i, chunk in enumerate(chunks)
]

lc_embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": str(device)},
)

vectorstore = LangChainFAISS.from_documents(documents, lc_embeddings)
lc_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Built LangChain FAISS vectorstore with", len(documents), "documents")
_docs = lc_retriever.invoke(qa_dataset[0]["question"])
print("Retrieved", len(_docs), "documents for a sample query")

/tmp/ipykernel_3203137/239892107.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS as LangChainFAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Built LangChain FAISS vectorstore with 3 documents
Retrieved 3 documents for a sample query


In [17]:
# TODO: Load `google/flan-t5-small` as an encoder-decoder model.
# Implement a small helper function to generate text with FLAN-T5.
# Expected output is kept below as a reference.
from transformers import T5Tokenizer, T5ForConditionalGeneration

t5_model_name = "google/flan-t5-small"
t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_name)
t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name).to(device)
t5_model.eval()


def generate_with_flan_t5(prompt: str, max_new_tokens: int = 80) -> str:
    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        output_ids = t5_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
        )
    return t5_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


print(generate_with_flan_t5("Question: What is the capital of France?\nAnswer:"))

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

london


In [18]:
# TODO: Implement helper functions for conversational RAG.
# Convert chat history to text and rewrite short follow-up questions.
def chat_history_to_text(chat_history) -> str:
    lines = []
    for turn in chat_history:
        if isinstance(turn, (tuple, list)) and len(turn) == 2:
            role, content = turn
        else:
            content = getattr(turn, "content", str(turn))
            cls = turn.__class__.__name__.lower()
            role = "human" if "human" in cls else "ai"
        role = "User" if str(role).lower() in ("human", "user") else "Assistant"
        lines.append(f"{role}: {content}")
    return "\n".join(lines)


def rewrite_question(question: str, chat_history) -> str:
    if not chat_history:
        return question
    history_text = chat_history_to_text(chat_history)
    prompt = (
        "Given the conversation history and a follow-up question, rewrite the "
        "follow-up question to be a standalone question that can be understood "
        "without the history. Only output the rewritten question.\n\n"
        f"Chat history:\n{history_text}\n\n"
        f"Follow-up question: {question}\n"
        "Standalone question:"
    )
    rewritten = generate_with_flan_t5(prompt, max_new_tokens=64).strip()
    return rewritten if rewritten else question

In [19]:
# TODO: Implement `SimpleConversationalRAG`.
# Steps: rewrite follow-up question, retrieve documents, generate answer.
class SimpleConversationalRAG:
    def __init__(self, retriever):
        self.retriever = retriever

    def invoke(self, inputs: Dict) -> Dict:
        question = inputs["input"]
        chat_history = inputs.get("chat_history", [])

        standalone_question = rewrite_question(question, chat_history)

        docs = self.retriever.invoke(standalone_question)
        retrieved_context = "\n\n".join(d.page_content for d in docs)

        prompt = (
            "Answer the question using only the context below.\n\n"
            f"Context:\n{retrieved_context}\n\n"
            f"Question: {standalone_question}\n"
            "Answer:"
        )
        answer = generate_with_flan_t5(prompt, max_new_tokens=80)

        return {
            "input": question,
            "standalone_question": standalone_question,
            "answer": answer,
            "retrieved_context": retrieved_context,
        }


rag_chain_with_memory = SimpleConversationalRAG(lc_retriever)

In [20]:
# Test the corrected memory-based RAG pipeline.
chat_history = []

question_1 = "What was Super Bowl 50?"
response_1 = rag_chain_with_memory.invoke({
    "input": question_1,
    "chat_history": chat_history
})

print("Question 1:", question_1)
print("Standalone question 1:", response_1["standalone_question"])
print("Answer 1:", response_1["answer"])

chat_history.append(("human", question_1))
chat_history.append(("ai", response_1["answer"]))

question_2 = "Where was it played?"
response_2 = rag_chain_with_memory.invoke({
    "input": question_2,
    "chat_history": chat_history
})

print("\nQuestion 2:", question_2)
print("Standalone question 2:", response_2["standalone_question"])
print("Answer 2:", response_2["answer"])
print("\nRetrieved context for turn 2:")
print(response_2["retrieved_context"][:1000], "...")


Question 1: What was Super Bowl 50?
Standalone question 1: What was Super Bowl 50?
Answer 1: an American football game

Question 2: Where was it played?
Standalone question 2: Where was it played?
Answer 2: Levi's Stadium

Retrieved context for turn 2:
arolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as w

ersary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.

Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) 

In [21]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []


def ask_question(query: str):
    response = rag_chain_with_memory.invoke({
        "input": query,
        "chat_history": chat_history
    })

    # Update chat history after each turn.
    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=response["answer"]))
    return response


# First turn: a real SQuAD-style question.
q1 = qa_dataset[0]["question"]
response1 = ask_question(q1)
print("User:", q1)
print("Standalone:", response1["standalone_question"])
print("Assistant:", response1["answer"])

# Follow-up question that depends on the previous topic.
q2 = "Which team represented the NFC in that game?"
response2 = ask_question(q2)
print("\nUser:", q2)
print("Standalone:", response2["standalone_question"])
print("Assistant:", response2["answer"])


User: Which NFL team represented the AFC at Super Bowl 50?
Standalone: Which NFL team represented the AFC at Super Bowl 50?
Assistant: Denver Broncos

User: Which team represented the NFC in that game?
Standalone: Which team represented the NFC in that game?
Assistant: Carolina Panthers


### Explanation for Question 6

A history-aware retriever first checks whether the latest user question is self-contained. If the question contains references such as *it*, *that game*, or *the previous answer*, the system uses the chat history to rewrite it into a standalone query. For example:

```text
Chat history: What was Super Bowl 50?
Follow-up: Where was it played?
Standalone query: Where was Super Bowl 50 played?
```

The retriever should usually receive this standalone query rather than the whole chat history. Passing the entire chat history directly to the retriever can introduce irrelevant words from previous turns and make vector search less focused. A short standalone query preserves the missing context while keeping retrieval precise.

In this notebook, we implement the history-aware logic explicitly instead of relying on `langchain.chains`, because those imports are version-sensitive. The idea is the same: use memory to rewrite the query, retrieve relevant documents, then answer from the retrieved context.


## Part 8: RAG with an Encoder-Decoder Generator

So far, the retriever has been encoder-based, but the generator has been GPT‑2, which is a decoder-only language model.

In this part, we replace GPT‑2 with an encoder-decoder model, **google/flan-t5-small**. The retrieved context and the user question are passed to the encoder, and the decoder generates the answer.

### Question 7
Implement a RAG pipeline that uses the same retriever as before, but replaces GPT‑2 with an encoder-decoder model such as `google/flan-t5-small`.

1. Compare the GPT‑2-based RAG and the T5-based RAG on at least two questions.
2. Explain how the retrieved context is used differently in GPT‑2 and T5.
3. Why is T5 considered an encoder-decoder model, while GPT‑2 is decoder-only?

In [22]:
# TODO: Implement the T5-based RAG pipeline.
# Use the same retriever, but replace GPT-2 with FLAN-T5.
# Expected output is kept below as a reference.
class T5RAGSystem:
    def __init__(self, index, embed_model, chunks, t5_tokenizer, t5_model):
        self.index = index
        self.embed_model = embed_model
        self.chunks = chunks
        self.t5_tokenizer = t5_tokenizer
        self.t5_model = t5_model

    def query(self, question: str, top_k: int = 3, max_new_tokens: int = 80) -> Dict:
        retrieved = retrieve(question, self.index, self.embed_model, self.chunks, top_k=top_k)
        context_chunks = [c for c, _ in retrieved]
        context = "\n\n".join(context_chunks) if context_chunks else "(no context retrieved)"
        prompt = (
            "Answer the question based on the context.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\n"
            "Answer:"
        )
        answer = generate_with_flan_t5(prompt, max_new_tokens=max_new_tokens)
        return {
            "question": question,
            "answer": answer,
            "retrieved_chunks": context_chunks,
        }


t5_rag = T5RAGSystem(index, embed_model, chunks, t5_tokenizer, t5_model)

_test = qa_dataset[0]["question"]
_res = t5_rag.query(_test, top_k=3)
print("Q:", _test)
print("T5 A:", _res["answer"])

Q: Which NFL team represented the AFC at Super Bowl 50?
T5 A: Denver Broncos


In [23]:
# TODO: Compare GPT-2 RAG and T5 RAG on at least two questions.
# Use the same retrieved context for both models for a fair comparison.
# Expected output is kept below as a reference.
questions_to_compare = [qa_dataset[0]["question"], qa_dataset[1]["question"]]

for q in questions_to_compare:
    retrieved = retrieve(q, index, embed_model, chunks, top_k=3)
    context_chunks = [c for c, _ in retrieved]
    context = "\n\n".join(context_chunks) if context_chunks else "(no context retrieved)"

    gpt2_prompt = build_rag_prompt(context_chunks, q)
    gpt2_answer = generate_answer(gpt2_prompt, max_new_tokens=80)

    t5_prompt = (
        "Answer the question based on the context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {q}\n"
        "Answer:"
    )
    t5_answer = generate_with_flan_t5(t5_prompt, max_new_tokens=80)

    print("Question:", q)
    print("GPT-2 RAG:", gpt2_answer)
    print("T5 RAG   :", t5_answer)
    print("-" * 70)

Question: Which NFL team represented the AFC at Super Bowl 50?
GPT-2 RAG: The Denver Broncos.
T5 RAG   : Denver Broncos
----------------------------------------------------------------------
Question: Which NFL team represented the NFC at Super Bowl 50?
GPT-2 RAG: The Denver Broncos.
T5 RAG   : Denver Broncos
----------------------------------------------------------------------


### Explanation for Question 7

In both systems, the retriever is exactly the same. The question is embedded using the SentenceTransformer model, and FAISS retrieves the top-k most relevant chunks from the real SQuAD context corpus.

The main difference is the generator architecture.

In the **GPT‑2-based RAG system**, the retrieved context, the question, and the `Answer:` prefix are concatenated into one prompt. GPT‑2 is a **decoder-only** model, so it generates the answer by continuing this prompt from left to right. During generation, each new token can attend only to previous tokens, including the retrieved context and the question.

In the **T5-based RAG system**, the retrieved context and the question are passed to the **encoder**. The encoder reads the full input sequence and produces contextual representations. Then the **decoder** generates the answer autoregressively while attending to the encoder outputs through cross-attention. Therefore, the retrieved context is used as an encoded source sequence rather than only as a text prefix.

T5 is considered an **encoder-decoder** model because it has two separate Transformer components: an encoder for processing the input text and a decoder for generating the output text. GPT‑2 is **decoder-only** because it only contains the autoregressive Transformer decoder stack and generates text by predicting the next token from the previous tokens.

In practice, FLAN‑T5 usually gives cleaner answers for this task than base GPT‑2 because FLAN‑T5 is instruction-tuned and better aligned with question answering. Base GPT‑2 is mainly trained for next-token prediction, so it may continue the prompt instead of directly answering the question.

## Summary

You have:
- Built a **stateless RAG system** from real SQuAD contexts, embeddings, FAISS, and GPT‑2.
- Observed its weakness on follow-up questions.
- Used a LangChain FAISS vectorstore and an explicit **history-aware query rewriting** step to add conversational memory.
- Replaced the unstable GPT‑2 memory generator with **FLAN‑T5** for reformulation and answering.
- Replaced the decoder-only GPT‑2 generator with an **encoder-decoder FLAN‑T5** generator.
- Compared GPT‑2-based RAG and T5-based RAG on the same retrieved contexts.

### Final Theoretical Questions
1. Compare the stateless and conversational RAG pipelines. What are the main differences in the retrieval step?
2. In the conversational version, where is the ‘memory’ actually stored?
3. How could you extend this system to handle **multiple users**? What would you need to change?
4. Suppose you wanted to replace GPT‑2 with a model that does not support the “reformulate a standalone question” task well. What alternatives could you explore?
5. Why can an encoder-decoder model such as T5 be more suitable for question answering than a base decoder-only model such as GPT‑2?

### References
- [SQuAD: 100,000+ Questions for Machine Comprehension of Text](https://rajpurkar.github.io/SQuAD-explorer/)
- [Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (2020)](https://arxiv.org/abs/2005.11401)
- [FAISS library](https://github.com/facebookresearch/faiss)
- [FLAN-T5 model family](https://huggingface.co/google/flan-t5-small)


### Answers to Final Theoretical Questions

1. In the stateless pipeline the retriever embeds the raw question directly. In the conversational pipeline the question is first rewritten into a standalone question using the chat history, and that rewritten question is what gets embedded and searched, so follow-ups retrieve the right context.

2. The memory lives in the chat_history list that we keep in Python and pass into the chain on every turn. The models themselves remember nothing between calls.

3. To support many users you would keep a separate chat_history for each one, keyed by a user or session id, instead of one global list. The retriever, embeddings, and generator can stay shared, but each user's conversation state has to be isolated.

4. If the generator cannot reformulate well, you could use a separate small instruction-tuned model just for the rewriting step, replace learned rewriting with a simple rule that appends recent history to the question, or use a dedicated query-rewriting model so the weak generator only has to answer.

5. T5 is an encoder-decoder that reads the whole context with its encoder and then generates the answer with cross-attention, and FLAN-T5 is instruction-tuned for tasks like question answering. Base GPT-2 only predicts the next token and is not instruction-tuned, so it often just continues the prompt instead of directly answering the question.